In [ ]:
import tensorflow as tf
import os
import cv2
import numpy as np
from matplotlib import pyplot as plt
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing import image
from sklearn.metrics import confusion_matrix, classification_report
from keras.utils import to_categorical
from keras.callbacks import ReduceLROnPlateau

In [ ]:
data_dir = os.path.join('drive', 'MyDrive', 'data')

##CREATE DATASET MANUALLY

In [ ]:
labels = ['not_ripples', 'transition', 'ripples']
img_size = 100
def get_data(data_dir):
    data = []
    for label in labels:
        path = os.path.join(data_dir, label)
        class_num = labels.index(label)
        for img in os.listdir(path):
            try:
                img_arr = cv2.imread(os.path.join(path, img))[...,::-1] #convert BGR to RGB format
                resized_arr = cv2.resize(img_arr, (img_size, img_size)) # Reshaping images to preferred size
                data.append([resized_arr, class_num])
            except Exception as e:
                print(e)
    return data

Dataset for first classification

In [ ]:
train = get_data(os.path.join(data_dir, 'Lines_for_train', '20x20', 'Lineas_Sonarwiz', 'train'))
val = get_data(os.path.join(data_dir, 'Lines_for_train', '20x20', 'Lineas_Sonarwiz', 'test'))

In [ ]:
x_train = []
y_train = []
x_val = []
y_val = []

for feature, label in train:
  x_train.append(feature)
  y_train.append(label)

for feature, label in val:
  x_val.append(feature)
  y_val.append(label)

# Normalize the data
x_train = np.array(x_train)
x_val = np.array(x_val)

x_train.reshape(-1, img_size, img_size, 1)
y_train = np.array(y_train)

x_val.reshape(-1, img_size, img_size, 1)
y_val = np.array(y_val)

In [ ]:
x_train,x_val,y_train,y_val=train_test_split(x_train,y_train, test_size=0.3)
y_train=to_categorical(y_train)
y_val=to_categorical(y_val)

Dataset for fourier classification

In [ ]:
train_FT = get_data(os.path.join(data_dir, 'Lines_for_train', '20x20', 'Lineas_Sonarwiz', 'ft', 'train'))
val_FT = get_data(os.path.join(data_dir, 'Lines_for_train', '20x20', 'Lineas_Sonarwiz', 'ft', 'test'))

In [ ]:
x_train_FT = []
y_train_FT = []
x_val_FT = []
y_val_FT = []

for feature, label in train_FT:
  x_train_FT.append(feature)
  y_train_FT.append(label)

for feature, label in val_FT:
  x_val_FT.append(feature)
  y_val_FT.append(label)

# Normalize the data
x_train_FT = np.array(x_train_FT)
x_val_FT = np.array(x_val_FT)

x_train_FT.reshape(-1, img_size, img_size, 1)
y_train_FT = np.array(y_train_FT)

x_val_FT.reshape(-1, img_size, img_size, 1)
y_val_FT = np.array(y_val_FT)

In [ ]:
x_train_FT,x_val_FT,y_train_FT,y_val_FT=train_test_split(x_train_FT,y_train_FT, test_size=0.3)
y_train_FT=to_categorical(y_train_FT)
y_val_FT=to_categorical(y_val_FT)

Dataset for Surface Roughness classification

In [ ]:
train_SR = get_data(os.path.join(data_dir, 'Lines_for_train', '20x20', 'Lineas_Sonarwiz', 'attribute', 'train'))
val_SR = get_data(os.path.join(data_dir, 'Lines_for_train', '20x20', 'Lineas_Sonarwiz', 'attribute', 'test'))

In [ ]:
x_train_SR = []
y_train_SR = []
x_val_SR = []
y_val_SR = []

for feature, label in train_SR:
  x_train_SR.append(feature)
  y_train_SR.append(label)

for feature, label in val_SR:
  x_val_SR.append(feature)
  y_val_SR.append(label)

# Normalize the data
x_train_SR = np.array(x_train_SR)
x_val_SR = np.array(x_val_SR)

x_train_SR.reshape(-1, img_size, img_size, 1)
y_train_SR = np.array(y_train_SR)

x_val_SR.reshape(-1, img_size, img_size, 1)
y_val_SR = np.array(y_val_SR)

In [ ]:
x_train_SR,x_val_SR,y_train_SR,y_val_SR=train_test_split(x_train_SR,y_train_SR, test_size=0.3)
y_train_SR=to_categorical(y_train_SR)
y_val_SR=to_categorical(y_val_SR)

##CREATING FOLDER FOR LOGS

In [ ]:
logdir= os.path.join(data_dir, 'logs')

tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir)

##USING EXISTING NEURAL NETWORK MODELS

In [ ]:
from tensorflow.keras.applications import ResNet50, efficientnet_v2, VGG16, DenseNet121

In [ ]:
# Initialize the Pretrained Model
#--------------------ResNet50-------------------
ResNet50_feature_extractor_20by20 = ResNet50(weights='imagenet', input_shape=(100, 100, 3), include_top=False)
#----------------EfficientNetV2S----------------
ENV2S_feature_extractor_20by20 = efficientnet_v2.EfficientNetV2S(weights='imagenet', input_shape=(100, 100, 3), include_top=False, include_preprocessing=True)
#---------------------VGG16---------------------
VGG16_feature_extractor_20by20 = VGG16(weights='imagenet', input_shape=(100, 100, 3), include_top=False)
#------------------DenseNet121------------------
DenseNet121_feature_extractor_20by20 = DenseNet121(weights='imagenet', input_shape=(100, 100, 3), include_top=False)

In [ ]:
# Set this parameter to make sure it's not being trained
#--------------------ResNet50-------------------
ResNet50_feature_extractor_20by20.trainable = False
#----------------EfficientNetV2S----------------
ENV2S_feature_extractor_20by20.trainable = False
#---------------------VGG16---------------------
VGG16_feature_extractor_20by20.trainable = False
#------------------DenseNet121------------------
DenseNet121_feature_extractor_20by20.trainable = False

# Set the input layer
#--------------------ResNet50-------------------
ResNet50_input20by20_ = tf.keras.Input(shape=(100, 100, 3))
#----------------EfficientNetV2S----------------
ENV2S_input20by20_ = tf.keras.Input(shape=(100, 100, 3))
#---------------------VGG16---------------------
VGG16_input20by20_ = tf.keras.Input(shape=(100, 100, 3))
#------------------DenseNet121------------------
DenseNet121_input20by20_ = tf.keras.Input(shape=(100, 100, 3))

# Set the feature extractor layer
#--------------------ResNet50-------------------
ResNet50_x20by20 = ResNet50_feature_extractor_20by20(ResNet50_input20by20_, training=False)
#----------------EfficientNetV2S----------------
ENV2S_x20by20 = ENV2S_feature_extractor_20by20(ENV2S_input20by20_, training=False)
#---------------------VGG16---------------------
VGG16_x20by20 = VGG16_feature_extractor_20by20(VGG16_input20by20_, training=False)
#------------------DenseNet121------------------
DenseNet121_x20by20 = DenseNet121_feature_extractor_20by20(DenseNet121_input20by20_, training=False)

# Set the pooling layer
#--------------------ResNet50-------------------
ResNet50_x20by20 = tf.keras.layers.GlobalAveragePooling2D()(ResNet50_x20by20)
#----------------EfficientNetV2S----------------
ENV2S_x20by20 = tf.keras.layers.GlobalAveragePooling2D()(ENV2S_x20by20)
#---------------------VGG16---------------------
VGG16_x20by20 = tf.keras.layers.GlobalAveragePooling2D()(VGG16_x20by20)
#------------------DenseNet121------------------
DenseNet121_x20by20 = tf.keras.layers.GlobalAveragePooling2D()(DenseNet121_x20by20)
'''---------------FOURIER TRANSFORM---------------'''
#--------------------ResNet50-------------------
ResNet50_x20by20_FT = tf.keras.layers.GlobalAveragePooling2D()(ResNet50_x20by20)
#----------------EfficientNetV2S----------------
ENV2S_x20by20_FT = tf.keras.layers.GlobalAveragePooling2D()(ENV2S_x20by20)
'''---------------SURFACE ROUGHNESS---------------'''
#--------------------ResNet50-------------------
ResNet50_x20by20_SR = tf.keras.layers.GlobalAveragePooling2D()(ResNet50_x20by20)
#----------------EfficientNetV2S----------------
ENV2S_x20by20_SR = tf.keras.layers.GlobalAveragePooling2D()(ENV2S_x20by20)
#---------------------VGG16---------------------
VGG16_x20by20_SR = tf.keras.layers.GlobalAveragePooling2D()(VGG16_x20by20)
#------------------DenseNet121------------------
DenseNet121_x20by20_SR = tf.keras.layers.GlobalAveragePooling2D()(DenseNet121_x20by20)

# Set layer with dropout rate function
#--------------------ResNet50-------------------
ResNet50_x20by20 = tf.keras.layers.Dropout(0.2)(ResNet50_x20by20)
#----------------EfficientNetV2S----------------
ENV2S_x20by20 = tf.keras.layers.Dropout(0.2)(ENV2S_x20by20)
#---------------------VGG16---------------------
VGG16_x20by20 = tf.keras.layers.Dropout(0.2)(VGG16_x20by20)
#------------------DenseNet121------------------
DenseNet121_x20by20 = tf.keras.layers.Dropout(0.2)(DenseNet121_x20by20)
'''---------------FOURIER TRANSFORM---------------'''
#--------------------ResNet50-------------------
ResNet50_x20by20_FT = tf.keras.layers.Dropout(0.2)(ResNet50_x20by20_FT)
#----------------EfficientNetV2S----------------
ENV2S_x20by20_FT = tf.keras.layers.Dropout(0.2)(ENV2S_x20by20_FT)
'''---------------SURFACE ROUGHNESS---------------'''
#--------------------ResNet50-------------------
ResNet50_x20by20_SR = tf.keras.layers.Dropout(0.2)(ResNet50_x20by20_SR)
#----------------EfficientNetV2S----------------
ENV2S_x20by20_SR = tf.keras.layers.Dropout(0.2)(ENV2S_x20by20_SR)
#---------------------VGG16---------------------
VGG16_x20by20_SR = tf.keras.layers.Dropout(0.2)(VGG16_x20by20_SR)
#------------------DenseNet121------------------
DenseNet121_x20by20_SR = tf.keras.layers.Dropout(0.2)(DenseNet121_x20by20_SR)

# Set first dense layer with sigmoid activation function
#--------------------ResNet50-------------------
ResNet50_x20by20 = tf.keras.layers.Dense(50, activation='sigmoid')(ResNet50_x20by20)
#----------------EfficientNetV2S----------------
ENV2S_x20by20 = tf.keras.layers.Dense(50, activation='sigmoid')(ENV2S_x20by20)
#---------------------VGG16---------------------
VGG16_x20by20 = tf.keras.layers.Dense(50, activation='sigmoid')(VGG16_x20by20)
#------------------DenseNet121------------------
DenseNet121_x20by20 = tf.keras.layers.Dense(50, activation='sigmoid')(DenseNet121_x20by20)
'''---------------FOURIER TRANSFORM---------------'''
#--------------------ResNet50-------------------
ResNet50_x20by20_FT = tf.keras.layers.Dense(50, activation='sigmoid')(ResNet50_x20by20_FT)
#----------------EfficientNetV2S----------------
ENV2S_x20by20_FT = tf.keras.layers.Dense(50, activation='sigmoid')(ENV2S_x20by20_FT)
'''---------------SURFACE ROUGHNESS---------------'''
#--------------------ResNet50-------------------
ResNet50_x20by20_SR = tf.keras.layers.Dense(50, activation='sigmoid')(ResNet50_x20by20_SR)
#----------------EfficientNetV2S----------------
ENV2S_x20by20_SR = tf.keras.layers.Dense(50, activation='sigmoid')(ENV2S_x20by20_SR)
#---------------------VGG16---------------------
VGG16_x20by20_SR = tf.keras.layers.Dense(50, activation='sigmoid')(VGG16_x20by20_SR)
#------------------DenseNet121------------------
DenseNet121_x20by20_SR = tf.keras.layers.Dense(50, activation='sigmoid')(DenseNet121_x20by20_SR)

# Set the final layer with softmax activation function
#--------------------ResNet50-------------------
ResNet50_output20by20_ = tf.keras.layers.Dense(3, activation='softmax')(ResNet50_x20by20)
#----------------EfficientNetV2S----------------
ENV2S_output20by20_ = tf.keras.layers.Dense(3, activation='softmax')(ENV2S_x20by20)
#---------------------VGG16---------------------
VGG16_output20by20_ = tf.keras.layers.Dense(3, activation='softmax')(VGG16_x20by20)
#------------------DenseNet121------------------
DenseNet121_output20by20_ = tf.keras.layers.Dense(3, activation='softmax')(DenseNet121_x20by20)
'''---------------FOURIER TRANSFORM---------------'''
#--------------------ResNet50-------------------
ResNet50_output20by20_FT = tf.keras.layers.Dense(3, activation='softmax')(ResNet50_x20by20_FT)
#----------------EfficientNetV2S----------------
ENV2S_output20by20_FT = tf.keras.layers.Dense(3, activation='softmax')(ENV2S_x20by20_FT)
'''---------------SURFACE ROUGHNESS---------------'''
#--------------------ResNet50-------------------
ResNet50_output20by20_SR = tf.keras.layers.Dense(3, activation='softmax')(ResNet50_x20by20_SR)
#----------------EfficientNetV2S----------------
ENV2S_output20by20_SR = tf.keras.layers.Dense(3, activation='softmax')(ENV2S_x20by20_SR)
#---------------------VGG16---------------------
VGG16_output20by20_SR = tf.keras.layers.Dense(3, activation='softmax')(VGG16_x20by20_SR)
#------------------DenseNet121------------------
DenseNet121_output20by20_SR = tf.keras.layers.Dense(3, activation='softmax')(DenseNet121_x20by20_SR)

In [ ]:
# Create the new model object
#--------------------ResNet50-------------------
ResNet50_model_20by20 = tf.keras.Model(ResNet50_input20by20_, ResNet50_output20by20_)
#----------------EfficientNetV2S----------------
ENV2S_model_20by20 = tf.keras.Model(ENV2S_input20by20_, ENV2S_output20by20_)
#---------------------VGG16---------------------
VGG16_model_20by20 = tf.keras.Model(VGG16_input20by20_, VGG16_output20by20_)
#------------------DenseNet121------------------
DenseNet121_model_20by20 = tf.keras.Model(DenseNet121_input20by20_, DenseNet121_output20by20_)

#FOURIER CLASSIFICATION
#--------------------ResNet50-------------------
ResNet50_model_20by20_FT = tf.keras.Model(ResNet50_input20by20_, ResNet50_output20by20_FT)
#----------------EfficientNetV2S----------------
ENV2S_model_20by20_FT = tf.keras.Model(ENV2S_input20by20_, ENV2S_output20by20_FT)

#SURFACE ROUGHNESS CLASSIFICATION
#--------------------ResNet50-------------------
ResNet50_model_20by20_SR = tf.keras.Model(ResNet50_input20by20_, ResNet50_output20by20_SR)
#----------------EfficientNetV2S----------------
ENV2S_model_20by20_SR = tf.keras.Model(ENV2S_input20by20_, ENV2S_output20by20_SR)
#---------------------VGG16---------------------
VGG16_model_20by20_SR = tf.keras.Model(VGG16_input20by20_, VGG16_output20by20_SR)
#------------------DenseNet121------------------
DenseNet121_model_20by20_SR = tf.keras.Model(DenseNet121_input20by20_, DenseNet121_output20by20_SR)

In [ ]:
opt_ResNet50 = tf.keras.optimizers.Adam(learning_rate=0.0001)
opt_ENV2S = tf.keras.optimizers.Adam(learning_rate=0.0001)
opt_VGG16 = tf.keras.optimizers.Adam(learning_rate=0.0001)
opt_DenseNet121 = tf.keras.optimizers.Adam(learning_rate=0.0001)

# Compile it
#--------------------ResNet50-------------------
ResNet50_model_20by20.compile(optimizer=opt_ResNet50, loss='categorical_crossentropy', metrics=['accuracy'])
#----------------EfficientNetV2S----------------
ENV2S_model_20by20.compile(optimizer=opt_ENV2S, loss='categorical_crossentropy', metrics=['accuracy'])
#---------------------VGG16---------------------
VGG16_model_20by20.compile(optimizer=opt_VGG16, loss='categorical_crossentropy', metrics=['accuracy'])
#------------------DenseNet121------------------
DenseNet121_model_20by20.compile(optimizer=opt_DenseNet121, loss='categorical_crossentropy', metrics=['accuracy'])

#FOURIER CLASSIFICATION
#--------------------ResNet50-------------------
ResNet50_model_20by20_FT.compile(optimizer=opt_ResNet50, loss='categorical_crossentropy', metrics=['accuracy'])
#----------------EfficientNetV2S----------------
ENV2S_model_20by20_FT.compile(optimizer=opt_ENV2S, loss='categorical_crossentropy', metrics=['accuracy'])

#SURFACE ROUGHNESS CLASSIFICATION
#--------------------ResNet50-------------------
ResNet50_model_20by20_SR.compile(optimizer=opt_ResNet50, loss='categorical_crossentropy', metrics=['accuracy'])
#----------------EfficientNetV2S----------------
ENV2S_model_20by20_SR.compile(optimizer=opt_ENV2S, loss='categorical_crossentropy', metrics=['accuracy'])
#---------------------VGG16---------------------
VGG16_model_20by20_SR.compile(optimizer=opt_VGG16, loss='categorical_crossentropy', metrics=['accuracy'])
#------------------DenseNet121------------------
DenseNet121_model_20by20_SR.compile(optimizer=opt_DenseNet121_SR, loss='categorical_crossentropy', metrics=['accuracy'])

#--------------------------------TRAINING MODEL FOR FIRST CLASSIFICATION--------------------------------

##TRAINING AND SAVING RESNET50 MODEL

Test with Adam optimizer, modified learning rate

In [ ]:
hist_ResNet50_20by20 = ResNet50_model_20by20.fit(x_train,y_train, epochs=75, validation_data=(x_val, y_val), callbacks=[tensorboard_callback])

In [ ]:
ResNet50_model_20by20.save(os.path.join(data_dir,'imageclassifier_3classes_ResNet50_20x20_75epoch.h5'))

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10,5))
ax[0].plot(hist_ResNet50_20by20.history['loss'], color='teal', label='loss')
ax[0].plot(hist_ResNet50_20by20.history['val_loss'], color='orange', label='val_loss')
ax[0].title.set_text('ResNet50 model 20x20 Adam optimizer Loss')
ax[0].legend(loc="upper right")
#ax[0].set_ylim([0, 1])

ax[1].plot(hist_ResNet50_20by20.history['accuracy'], color='teal', label='accurary')
ax[1].plot(hist_ResNet50_20by20.history['val_accuracy'], color='orange', label='val_accuracy')
ax[1].title.set_text('ResNet50 model 20x20 Adam optimizer Accuracy')
ax[1].legend(loc="lower right")
#ax[1].set_ylim([0.5, 1])

##TRAINING AND SAVING ENV2S MODEL

Test using Adam optimizer with learning rate modified values

In [ ]:
hist_ENV2S_20by20 = ENV2S_model_20by20.fit(x_train,y_train, epochs=200, validation_data=(x_val, y_val), callbacks=[tensorboard_callback])

In [ ]:
ENV2S_model_20by20.save(os.path.join(data_dir,'imageclassifier_3classes_ENV2S_20x20_200epoch.h5'))

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10,5))
ax[0].plot(hist_ENV2S_20by20.history['loss'], color='teal', label='loss')
ax[0].plot(hist_ENV2S_20by20.history['val_loss'], color='orange', label='val_loss')
ax[0].title.set_text('ENV2S model 20x20 Adam optimizer Loss')
ax[0].legend(loc="upper right")
#ax[0].set_ylim([0, 1])

ax[1].plot(hist_ENV2S_20by20.history['accuracy'], color='teal', label='accurary')
ax[1].plot(hist_ENV2S_20by20.history['val_accuracy'], color='orange', label='val_accuracy')
ax[1].title.set_text('ENV2S model 20x20 Adam optimizer Accuracy')
ax[1].legend(loc="lower right")
#ax[1].set_ylim([0.5, 1])

##TRAINING AND SAVING VGG-16 MODEL

Test using Adam optimizer, with learning rate modified values

In [ ]:
hist_VGG16_20by20 = VGG16_model_20by20.fit(x_train,y_train, epochs=200, validation_data=(x_val, y_val), callbacks=[tensorboard_callback])

In [ ]:
VGG16_model_20by20.save(os.path.join(data_dir,'imageclassifier_3classes_VGG16_20x20_200epoch.h5'))

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10,5))
ax[0].plot(hist_VGG16_20by20.history['loss'], color='teal', label='loss')
ax[0].plot(hist_VGG16_20by20.history['val_loss'], color='orange', label='val_loss')
ax[0].title.set_text('VGG16 model 20x20 Adam optimizer Loss')
ax[0].legend(loc="upper right")
#ax[0].set_ylim([0, 1])

ax[1].plot(hist_VGG16_20by20.history['accuracy'], color='teal', label='accurary')
ax[1].plot(hist_VGG16_20by20.history['val_accuracy'], color='orange', label='val_accuracy')
ax[1].title.set_text('VGG16 model 20x20 Adam optimizer Accuracy')
ax[1].legend(loc="lower right")
#ax[1].set_ylim([0.5, 1])

##TRAINING AND SAVING DENSETNET121 MODEL

Test with Adam optimizar, using learning rate modified values

In [ ]:
hist_DenseNet121_20by20 = DenseNet121_model_20by20.fit(x_train,y_train, epochs=200, validation_data=(x_val, y_val), callbacks=[tensorboard_callback])

In [ ]:
DenseNet121_model_20by20.save(os.path.join(data_dir,'imageclassifier_3classes_DenseNet121_20x20_200epoch.h5'))

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10,5))
ax[0].plot(hist_DenseNet121_20by20.history['loss'], color='teal', label='loss')
ax[0].plot(hist_DenseNet121_20by20.history['val_loss'], color='orange', label='val_loss')
ax[0].title.set_text('DenseNet121 model 20x20 Adam optimizer Loss')
ax[0].legend(loc="upper right")
#ax[0].set_ylim([0, 10])

ax[1].plot(hist_DenseNet121_20by20.history['accuracy'], color='teal', label='accurary')
ax[1].plot(hist_DenseNet121_20by20.history['val_accuracy'], color='orange', label='val_accuracy')
ax[1].title.set_text('DenseNet121 model 20x20 Adam optimizer Accuracy')
ax[1].legend(loc="lower right")
#ax[1].set_ylim([0.5, 1])

#------------------------------TRAINING MODEL FOR FOURIER CLASSIFICATION------------------------------

##TRAINING AND SAVING RESNET50 MODEL FOR FOURIER CLASSIFICATION

Test using Adam optimizer, using learning rate modified value

In [ ]:
hist_ResNet50_20by20_FT = ResNet50_model_20by20_FT.fit(x_train_FT, y_train_FT, epochs=70, validation_data=(x_val_FT, y_val_FT), callbacks = [tensorboard_callback], verbose = 1)

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10,5))
ax[0].plot(history.history['loss'], color='teal', label='loss')
ax[0].plot(history.history['val_loss'], color='orange', label='val_loss')
ax[0].title.set_text('ResNet50 FT model 20x20 Adam optimizer 0.001')
ax[0].legend(loc="upper right")
#ax[0].set_ylim([0, 10])

ax[1].plot(history.history['accuracy'], color='teal', label='accurary')
ax[1].plot(history.history['val_accuracy'], color='orange', label='val_accuracy')
ax[1].title.set_text('ResNet50 FT model 20x20 Adam optimizer Accuracy')
ax[1].legend(loc="lower right")
#ax[1].set_ylim([0.5, 1])

In [ ]:
ResNet50_model_20by20_FT.save(os.path.join(data_dir,'imageclassifier_3classes_ResNet50_20x20_70epoch_FT.h5'))

##TRAINING AND SAVING ENV2S MODEL FOR FOURIER CLASSIFICATION

Test with Adam optimizer, using learning rate modified value

In [ ]:
hist_ENV2S_20by20_FT = ENV2S_model_20by20_FT.fit(x_train_FT, y_train_FT, epochs=200, validation_data=(x_val_FT, y_val_FT), callbacks = [tensorboard_callback], verbose = 1)

In [ ]:
ENV2S_model_20by20_FT.save(os.path.join(data_dir,'imageclassifier_3classes_ENV2S_20x20_200epoch_FT.h5'))

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10,5))
ax[0].plot(hist_ENV2S_20by20_FT.history['loss'], color='teal', label='loss')
ax[0].plot(hist_ENV2S_20by20_FT.history['val_loss'], color='orange', label='val_loss')
ax[0].title.set_text('ENV2S FT model 20x20 Adam optimizer Loss - LR=Default Drop=Default')
ax[0].legend(loc="upper right")
#ax[0].set_ylim([0, 1])

ax[1].plot(hist_ENV2S_20by20_FT.history['accuracy'], color='teal', label='accurary')
ax[1].plot(hist_ENV2S_20by20_FT.history['val_accuracy'], color='orange', label='val_accuracy')
ax[1].title.set_text('ENV2S FT model 20x20 Adam optimizer Accuracy')
ax[1].legend(loc="lower right")
#ax[1].set_ylim([0.5, 1])

#-------------------TRAINING MODEL FOR SURFACE ROUGHNESS CLASSIFICATION-------------------

##TRAINING AND SAVING RESNET50 MODEL FOR SR CLASSIFICATION

Test with Adam optimizer, using learning rate modified values

In [ ]:
hist_ResNet50_20by20_SR = ResNet50_model_20by20_SR.fit(x_train_SR,y_train_SR, epochs=80, validation_data=(x_val_SR, y_val_SR), callbacks=[tensorboard_callback])

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10,5))
ax[0].plot(hist_ResNet50_20by20_SR.history['loss'], color='teal', label='loss')
ax[0].plot(hist_ResNet50_20by20_SR.history['val_loss'], color='orange', label='val_loss')
ax[0].title.set_text('ResNet50 SR model 20x20 Adam optimizer Loss - LR=Default Drop=Default')
ax[0].legend(loc="upper right")
#ax[0].set_ylim([0, 1])

ax[1].plot(hist_ResNet50_20by20_SR.history['accuracy'], color='teal', label='accurary')
ax[1].plot(hist_ResNet50_20by20_SR.history['val_accuracy'], color='orange', label='val_accuracy')
ax[1].title.set_text('ResNet50 SR model 20x20 Adam optimizer Accuracy')
ax[1].legend(loc="lower right")
#ax[1].set_ylim([0.5, 1])

In [ ]:
ResNet50_model_20by20_SR.save(os.path.join(data_dir,'imageclassifier_3classes_ResNet50_20x20_80epoch_SR.h5'))

##TRAINING AND SAVING VGG16 MODEL FOR SR CLASSIFICATION

Test with Adam optimizer, using learning rate modified values

In [ ]:
hist_VGG16_20by20_SR = VGG16_model_20by20_SR.fit(x_train_SR,y_train_SR, epochs=200, validation_data=(x_val_SR, y_val_SR), callbacks=[tensorboard_callback])

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10,5))
ax[0].plot(hist_VGG16_20by20_SR.history['loss'], color='teal', label='loss')
ax[0].plot(hist_VGG16_20by20_SR.history['val_loss'], color='orange', label='val_loss')
ax[0].title.set_text('VGG16 SR model 20x20 Adam optimizer Loss - LR=Default Drop=Default')
ax[0].legend(loc="upper right")
#ax[0].set_ylim([0, 1])

ax[1].plot(hist_VGG16_20by20_SR.history['accuracy'], color='teal', label='accurary')
ax[1].plot(hist_VGG16_20by20_SR.history['val_accuracy'], color='orange', label='val_accuracy')
ax[1].title.set_text('VGG16 SR model 20x20 Adam optimizer Accuracy')
ax[1].legend(loc="lower right")
#ax[1].set_ylim([0.5, 1])

In [ ]:
VGG16_model_20by20_SR.save(os.path.join(data_dir,'imageclassifier_3classes_VGG16_20x20_200epoch_SR.h5'))

##TRAINING AND SAVING ENV2S MODEL FOR SR CLASSIFICATION

Test with Adam optimizer, using learning rate modified values

In [ ]:
hist_ENV2S_20by20_SR = ENV2S_model_20by20.fit(x_train_SR,y_train_SR, epochs=200, validation_data=(x_val_SR, y_val_SR), callbacks=[tensorboard_callback])

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10,5))
ax[0].plot(hist_ENV2S_20by20_SR.history['loss'], color='teal', label='loss')
ax[0].plot(hist_ENV2S_20by20_SR.history['val_loss'], color='orange', label='val_loss')
ax[0].title.set_text('VGG16 SR model 20x20 Adam optimizer Loss - LR=Default Drop=Default')
ax[0].legend(loc="upper right")
#ax[0].set_ylim([0, 1])

ax[1].plot(hist_ENV2S_20by20_SR.history['accuracy'], color='teal', label='accurary')
ax[1].plot(hist_ENV2S_20by20_SR.history['val_accuracy'], color='orange', label='val_accuracy')
ax[1].title.set_text('ENV2S SR model 20x20 Adam optimizer Accuracy')
ax[1].legend(loc="lower right")
#ax[1].set_ylim([0.5, 1])

In [ ]:
ENV2S_model_20by20_SR.save(os.path.join(data_dir,'imageclassifier_3classes_ENV2S_20x20_200epoch_SR.h5'))

##TRAINING AND SAVING DENSENET121 MODEL FOR SR CLASSIFICATION

Test with Adam optimizer, using learning rate modified values

In [ ]:
hist_DenseNet121_20by20_SR = DenseNet121_model_20by20_SR.fit(x_train_SR,y_train_SR, epochs=120, validation_data=(x_val_SR, y_val_SR), callbacks=[tensorboard_callback])

In [ ]:
DenseNet121_model_20by20_SR.save(os.path.join(data_dir,'imageclassifier_3classes_DenseNet121_20x20_120epoch_SR.h5'))

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10,5))
ax[0].plot(hist_DenseNet121_20by20_SR.history['loss'], color='teal', label='loss')
ax[0].plot(hist_DenseNet121_20by20_SR.history['val_loss'], color='orange', label='val_loss')
ax[0].title.set_text('DenseNet121 FT model 20x20 Adam optimizer Loss - LR=Default Drop=Default')
ax[0].legend(loc="upper right")
#ax[0].set_ylim([0, 1])

ax[1].plot(hist_DenseNet121_20by20_SR.history['accuracy'], color='teal', label='accurary')
ax[1].plot(hist_DenseNet121_20by20_SR.history['val_accuracy'], color='orange', label='val_accuracy')
ax[1].title.set_text('DenseNet121 FT model 20x20 Adam optimizer Accuracy')
ax[1].legend(loc="lower right")
#ax[1].set_ylim([0.5, 1])